In [ ]:
# --- Config: paths & helpers (auto-inserted) ---
from pathlib import Path
import os, json
import pandas as pd

# Assume the notebook is opened from repository root (safe for most cases)
REPO = Path.cwd()
os.environ["PYTHONPATH"] = str(REPO)

# Core paths
META = REPO / "meta" / "master_metadata.csv"
LOGS = REPO / "logs"
LOGS_EF = REPO / "logs_ef"
LOGS_VOL = REPO / "logs_vol"
REPORTS = REPO / "reports"
REPORTS_SEG = REPO / "reports_seg"
REPORTS.mkdir(exist_ok=True); REPORTS_SEG.mkdir(exist_ok=True)

# --- Loaders ---
def load_seg_summaries():
    out = {}
    if LOGS.exists():
        for p in LOGS.glob("cv_seg_*_summary.json"):
            try:
                with open(p) as f:
                    out[p.stem] = json.load(f)
            except Exception:
                pass
    return out

def load_clf_summaries():
    out = {}
    p = LOGS / "cv_cls_summary.csv"
    if p.exists():
        out["all"] = pd.read_csv(p)
    p = LOGS_EF / "cv_cls_summary.csv"
    if p.exists():
        out["ef"] = pd.read_csv(p)
    p = LOGS_VOL / "cv_cls_summary.csv"
    if p.exists():
        out["vol"] = pd.read_csv(p)
    return out


# Cardiac Early Detection — Pipeline Notebook
Use this notebook to run the dataset processing and CV/HPO end-to-end.

In [ ]:
# Check environment
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| available:", torch.cuda.is_available())

In [ ]:
# Configure raw dataset locations (edit these paths)
RAW_CAMUS = "cardio_data/raw/camus"  # e.g., "/data/CAMUS"
RAW_ACDC  = "cardio_data/raw/acdc"   # e.g., "/data/ACDC"
SEED = 42

In [ ]:
# Process CAMUS (2D ED/ES)
import subprocess, sys
cmd = [sys.executable, "scripts/camus_process.py", "--raw", RAW_CAMUS, "--out", "cardio_data/processed/camus", "--size", "256"]
print("Running:", " ".join(cmd))
_ = subprocess.run(cmd, check=False)

In [ ]:
# Process ACDC (3D ED/ES)
import subprocess, sys
cmd = [sys.executable, "scripts/acdc_process.py", "--raw", RAW_ACDC, "--out", "cardio_data/processed/acdc", "--target_spacing", "1.25", "1.25", "10.0"]
print("Running:", " ".join(cmd))
_ = subprocess.run(cmd, check=False)

In [ ]:
# Make splits
import subprocess, sys
cmd = [sys.executable, "scripts/make_splits.py", "--meta", "meta/master_metadata.csv", "--seed", str(SEED)]
print("Running:", " ".join(cmd))
_ = subprocess.run(cmd, check=False)

In [ ]:
# Quick sanity check of metadata
import pandas as pd
df = pd.read_csv("meta/master_metadata.csv")
print(df.head())
print("\nCounts by dataset & split:")
print(df.groupby(["dataset","split"]).size())

In [ ]:
# Outer CV with nested HPO (CAMUS classification)
import subprocess, sys
cmd = [sys.executable, "scripts/torch_cv.py", "--meta", "meta/master_metadata.csv", "--view", "4CH", "--phase", "ED", "--folds", "5", "--seed", str(SEED), "--trials", "25"]
print("Running:", " ".join(cmd))
_ = subprocess.run(cmd, check=False)